In [ ]:
from datasets import load_dataset

ds = load_dataset("axmeu/BPE_WP_dataset")
train_texts = ds["train"]["text"][:500]
test_texts  = ds["test"]["text"][:50]

/home/onyxia/work/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys
sys.path.append("..")
from src.tokenizers.BPE.naive import BPE
from src.tokenizers.BPE.Fast_BPE import FastBPE


In [ ]:
bpe = BPE(vocab_size=20_000)
bpe.train(train_texts)

for text in test_texts:
    print(f"BPE: {bpe.encode(text)[:10]}")
    print()

Initial vocab size: 63
Unique words: 99,549
1000 merges... vocab size: 1063
2000 merges... vocab size: 2063
3000 merges... vocab size: 3063
4000 merges... vocab size: 4063
5000 merges... vocab size: 5063


In [4]:
fast_bpe = FastBPE(vocab_size=8_000)
fast_bpe.train(train_texts)

for text in test_texts:
    print(f"Fast BPE      : {fast_bpe.encode(text)[:10]}")

Initial vocab size: 37
Unique words: 85,923
1000 merges... vocab size: 1037
2000 merges... vocab size: 2037
3000 merges... vocab size: 3037
4000 merges... vocab size: 4037
5000 merges... vocab size: 5037
6000 merges... vocab size: 6037
7000 merges... vocab size: 7037
Training complete in 3.76s
Final vocab size: 8000
Fast BPE      : ['m', 'u', 'c', 'o', 'nate</w>', 'l', 'a', 'c', 't', 'o']
Fast BPE      : ['the</w>', 'a', 'nteri', 'or</w>', 'tr', 'i', 'angle</w>', 'i', 's</w>', 'a</w>']
Fast BPE      : ['the</w>', 'd', 'ean</w>', 'heritage</w>', 'c', 'e', 'n', 'tr', 'e</w>', 'i']
Fast BPE      : ['dur', 'b', 'a', 'n', 'ville</w>', 'i', 's</w>', 'a</w>', 'town</w>', 'i']
Fast BPE      : ['this</w>', 'i', 's</w>', 'a</w>', 'l', 'i', 'st</w>', 'of</w>', 'rivers</w>', 'of</w>']
Fast BPE      : ['b', 'all', 'y', 'm', 'ar', 'ti', 'n</w>', 'i', 's</w>', 'o']
Fast BPE      : ['r', 'ay</w>', 'i', 's</w>', 'a</w>', 's', 'o', 'ng</w>', 'b', 'y</w>']
Fast BPE      : ['christopher</w>', 'lee</w>', '

In [30]:
import pandas as pd
df = pd.read_csv("../data/Lexique4/Lexique4.tsv", sep="\t",
                 usecols=["1_Mot", "5_Cgram","31_MorphoStruct", "32_MorphoDecomp"])

In [ ]:
df = df.dropna(subset=["1_Mot", "5_Cgram", "31_MorphoStruct", "32_MorphoDecomp"])

df = df[df["5_Cgram"] == "VER"]
df = df[~df["32_MorphoDecomp"].str.contains(r"\[", regex=True)]

benchmark = (df[df["31_MorphoStruct"] != "0-1-0"]
               .groupby("31_MorphoStruct")
               .filter(lambda x: len(x) >= 100)
               .groupby("31_MorphoStruct")
               .apply(lambda x: x.sample(min(len(x), 100), random_state=42))
               .reset_index(level=0)
               .reset_index(drop=True))
benchmark = benchmark[~benchmark["32_MorphoDecomp"].str.contains(r"\[", regex=True)]
benchmark

,31_MorphoStruct,1_Mot,5_Cgram,32_MorphoDecomp
0,0-1-1,négociant,VER,/négoc(e).iant
1,0-1-1,tigré,VER,/tigr(e).é
2,0-1-1,blâme,VER,/blâm(er).e
3,0-1-1,aggloméré,VER,/agglomér(é).é
4,0-1-1,accabler,VER,/accabl(er).er
...,...,...,...,...
595,2-1-1,réassigner,VER,_ré_a(d)/s/sign(e).er
596,2-1-1,désemparer,VER,_dés_em/par(er).er
597,2-1-1,réengager,VER,_ré_en/gag(e).er
598,2-1-1,réencadrer,VER,_ré_en/cadr(e).er


In [ ]:
import re

def parse_morph(decomp: str) -> list(str):
    clean = decomp.replace("_", "/").replace("{", "").replace("}", "")
    morphemes = re.split(r"[/.]", clean)
    morphemes = [re.sub(r"\(.*?\)", "", m).strip() for m in morphemes]
    return [m for m in morphemes if m]

# 596	2-1-1	réassigner	VER	_ré_a(d)/s/sign(e).er
parse_morph("_ré_a(d)/s/sign(e).er")

['ré', 'a', 's', 'sign', 'er']

In [71]:
from datasets import load_dataset

benchmark = load_dataset("axmeu/FrVMorpho")["train"].to_pandas()


In [72]:
benchmark.sample()

,31_MorphoStruct,1_Mot,5_Cgram,32_MorphoDecomp,morphemes
141,0-1-2,bombardé,VER,/bomb(e).ard(er).é,"[bomb, ard, é]"
